# MATH-3515: Numerical Analysis
## Week 2 Exploration Notebook — Computer Arithmetic

This notebook accompanies the Week 2 handout and board lecture. Run each section as we reach it in class.

A few examples throughout draw on basketball, football, and soccer stats, and on
computer science / business scenarios, to connect the math to things you already
think about. Sections marked **(Optional)** are for after-class exploration and
are not part of the 70-minute in-class flow.

**Sections (≈70 minutes of core content, plus optional extensions):**

0. Setup
1. Opening Puzzle — The Harmonic Sum
2. Base Conversions — Verify by Hand
3. Flops — Counting and Timing
4. The Classic Example: `0.1 + 0.2` vs `0.3`
5. Normal Floats — Decomposing a Double
6. Worked Example: −12.625, Full Encode/Decode
7. Non-Normal Floats
8. Machine Epsilon, UFL, OFL
9. Errors — Absolute and Relative
10. *(Optional)* Arbitrary-Precision Arithmetic — Exact Sums for Business
11. Closing — Resolving the Opening Puzzle


### 0. Setup

Run this cell first.

In [ ]:
import sys
import struct
import math
import time
from fractions import Fraction
from decimal import Decimal
import numpy as np

print("Python float info:")
print(sys.float_info)


## 1. Opening Puzzle: The Harmonic Sum

Consider the sum $\displaystyle\sum_{k=1}^{n} \frac{1}{k}$.

- $S(n)$: add the terms from **largest to smallest** (i.e., $k = 1, 2, \dots, n$)
- $s(n)$: add the terms from **smallest to largest** (i.e., $k = n, n-1, \dots, 1$)

In exact arithmetic, $S(n) = s(n)$ always. Let's check what happens on a computer.

In [ ]:
def S_forward(n):
    """Sum 1/k from k=1 to n (largest term first)."""
    total = 0.0
    for k in range(1, n + 1):
        total += 1.0 / k
    return total


def s_reverse(n):
    """Sum 1/k from k=n down to 1 (smallest term first)."""
    total = 0.0
    for k in range(n, 0, -1):
        total += 1.0 / k
    return total


In [ ]:
print(f"{'n':>10} | {'S(n) - s(n)':>20}")
print("-" * 33)
for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    diff = S_forward(n) - s_reverse(n)
    print(f"{n:>10,} | {diff:>20.6e}")


**Compare this to the table from the handout.** Do your numbers match what we found on the board?

We'll come back to *why* this happens at the end of the notebook (Section 11).

## 2. Base Conversions — Verify by Hand

Let's turn the board algorithms (division-remainder for integers, multiply-by-2 for fractions) into code, and check them against the examples we did by hand.

In [ ]:
def to_binary_int(n):
    """Convert a nonnegative integer to a binary string using the division-remainder method."""
    if n == 0:
        return "0"
    bits = []
    while n > 0:
        n, rem = divmod(n, 2)
        bits.append(str(rem))
    return "".join(reversed(bits))


def to_binary_frac(f, max_bits=20):
    """Convert a fractional value 0 <= f < 1 to binary using the multiply-by-2 method.
    Stops early if f becomes exactly 0, otherwise stops after max_bits (safety limit)."""
    bits = []
    count = 0
    while f != 0 and count < max_bits:
        f *= 2
        bit = int(f)
        bits.append(str(bit))
        f -= bit
        count += 1
    return "".join(bits)


In [ ]:
# Integer example: 23 (base 10) -> base 2
print("23 (base 10) =", to_binary_int(23), "(base 2)")
print("Check: int('10111', 2) =", int("10111", 2))


In [ ]:
# Terminating fraction: 0.75 (base 10) -> base 2
print("0.75 (base 10) = 0." + to_binary_frac(0.75), "(base 2)")


In [ ]:
# Non-terminating fraction: 0.1 (base 10) -> base 2
print("0.1 (base 10) = 0." + to_binary_frac(0.1, max_bits=20) + "...", "(base 2)")
print()
print("Notice: this never actually reaches f = 0 within 20 bits.")
print("This is exactly why 0.1 cannot be stored exactly in binary floating point.")


**Extension:** the same functions work on any integer or fraction — including numbers from sports. A soccer match runs 90 minutes; a common jersey number is 12.

In [ ]:
# Extension: sports numbers through the same conversion functions
print("90 minutes (a full soccer match)  =", to_binary_int(90), "(base 2)")
print("Jersey number 12                  =", to_binary_int(12), "(base 2)")

# A free-throw percentage is a terminating fraction (3 out of 4 makes = 0.75)
print("0.75 free-throw rate (3-for-4)     = 0." + to_binary_frac(0.75), "(base 2)")


## 3. Flops — Counting and Timing

The number of floating-point operations (flops) an algorithm performs affects how fast it runs.

Suppose you're a sports-analytics or business analyst asked to total up a large column
of numbers — points scored on every play across a full season of games, or every
transaction amount in a company's ledger. Let's compare a loop-based sum against a
vectorized (NumPy) sum for the same computation.

In [ ]:
n = 2_000_000  # e.g., "one value per play/transaction, at league/company scale"
values = np.random.rand(n)

# Loop-based sum (pure Python)
def loop_sum(arr):
    total = 0.0
    for x in arr:
        total += x
    return total

start = time.perf_counter()
result_loop = loop_sum(values)
time_loop = time.perf_counter() - start

# Vectorized sum (NumPy)
start = time.perf_counter()
result_numpy = np.sum(values)
time_numpy = time.perf_counter() - start

print(f"Loop sum:   {result_loop:.6f}   time: {time_loop:.4f} s")
print(f"NumPy sum:  {result_numpy:.6f}   time: {time_numpy:.4f} s")
print(f"\nNumPy was about {time_loop / time_numpy:.0f}x faster for the same {n:,} additions.")


Both approaches perform roughly the same number of floating-point additions — the huge speed difference comes from *how* those flops are executed (vectorized, low-level CPU instructions vs. a Python-level loop), not from doing fewer of them.

This matters directly for **CS majors** writing performance-sensitive code, and for **business majors** working with large datasets in tools like Excel, SQL, or pandas: the same math done the "vectorized" way can be orders of magnitude faster.

## 4. The Classic Example: `0.1 + 0.2` vs `0.3`

We just saw that 0.1 cannot be represented exactly in binary. Let's see the consequence.

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)


`0.1`, `0.2`, and `0.3` are each already rounded to the nearest representable double when stored. Adding the rounded `0.1` and `0.2` does not necessarily land exactly on the rounded `0.3` — hence `False`.

**Takeaway:** never compare floats with `==`. Use a tolerance instead:

In [ ]:
print(math.isclose(0.1 + 0.2, 0.3))


## 5. Normal Floats — Decomposing a Double

Let's extract the sign, exponent, and significand bits directly from a real Python float, and verify the formula from the handout:

$$\text{value} = (-1)^{\sigma} \times \left(\frac{s}{b^{p-1}}\right) \times b^{e}$$

In [ ]:
def decode_double(x):
    """Decode a Python float (IEEE 754 double) into its sign, exponent, and significand."""
    [bits] = struct.unpack('>Q', struct.pack('>d', x))
    sign = (bits >> 63) & 0x1
    raw_exponent = (bits >> 52) & 0x7FF
    raw_significand = bits & ((1 << 52) - 1)

    bias = 1023
    true_exponent = raw_exponent - bias
    # value of the significand, including the implicit leading 1
    significand_value = 1 + raw_significand / (2 ** 52)

    reconstructed = ((-1) ** sign) * significand_value * (2.0 ** true_exponent)

    print(f"input value            : {x}")
    print(f"sign bit               : {sign}")
    print(f"raw exponent field     : {raw_exponent}  (biased)")
    print(f"true exponent (e)      : {true_exponent}")
    print(f"stored significand bits: {raw_significand:052b}")
    print(f"significand value      : {significand_value}")
    print(f"reconstructed value    : {reconstructed}")
    print(f"matches original?      : {reconstructed == x}")
    return sign, true_exponent, significand_value


In [ ]:
# Verify against the board example: -742.6801
decode_double(-742.6801)


**Extension:** try decoding a few real sports statistics — a basketball free-throw percentage, a football completion percentage, and a soccer expected-goals (xG) value.

In [ ]:
# Extension: decode a few sports stats the same way
decode_double(0.847)   # basketball free-throw percentage
decode_double(0.623)   # football completion percentage
decode_double(0.34)    # soccer expected goals (xG) in a match


## 6. Worked Example: −12.625, Full Encode/Decode

By hand, we found:

- $12.625_{10} = 1100.101_2$
- Normalized: $1.100101 \times 2^3$
- sign = 1, exponent = 3, significand = 1.100101 (binary) = 1.578125 (decimal)
- Stored (biased) exponent = 3 + 1023 = 1026

Let's confirm this with code.

In [ ]:
decode_double(-12.625)


In [ ]:
# Raw hex bytes, for comparison with the by-hand bit layout
print(struct.pack('>d', -12.625).hex())


**Check:** does the true exponent printed above match the by-hand value of 3? Does the significand value match $1.578125$ ($= 1.100101_2$)?

## 7. Non-Normal Floats

Special values and boundary behavior: overflow, underflow, infinity, and NaN.

A very common place these show up in practice: **sports statistics with a zero
denominator.** A rookie who hasn't played a game yet has no defined points-per-game;
a kicker with zero attempts has no defined field-goal percentage.

In [ ]:
print("Smallest positive normal float (UFL): ", sys.float_info.min)
print("Largest finite float (OFL):           ", sys.float_info.max)
print("Smallest positive subnormal float:    ", 5e-324)


In [ ]:
# Overflow
print("1e308 * 10  =", 1e308 * 10)

# Underflow (below the smallest subnormal)
print("5e-324 / 10 =", 5e-324 / 10)

# Indeterminate forms, framed as real sports-stat edge cases.
# (NumPy is used here instead of plain Python floats, since 0.0/0.0 on
#  plain Python floats raises ZeroDivisionError instead of returning nan.)
total_points = np.float64(0.0)
games_played = np.float64(0.0)
points_per_game = total_points / games_played
print("Points per game for a player with 0 games played:", points_per_game)

field_goals_made = np.float64(0.0)
field_goals_attempted = np.float64(0.0)
fg_percentage = field_goals_made / field_goals_attempted
print("Field-goal % for a kicker with 0 attempts:        ", fg_percentage)

print("inf - inf (an indeterminate form):                ", float('inf') - float('inf'))


## 8. Machine Epsilon, UFL, OFL — Confirm the Board Formulas

$$\varepsilon = b^{1-p} \qquad \text{UFL} = b^{L} \qquad \text{OFL} = b^{U+1}\,(1 - b^{-p})$$

For double precision: $b=2$, $p=53$, $L=-1022$, $U=1023$.

In [ ]:
b, p, L, U = 2, 53, -1022, 1023

epsilon = b ** (1 - p)
UFL = float(b) ** L
# Computed as (2 - 2^(1-p)) * 2^U instead of 2^(U+1)*(1-2^-p) directly,
# since forming 2**1024 as a standalone float overflows even though the
# final answer does not.
OFL = (2.0 - 2.0 ** (1 - p)) * (2.0 ** U)

print(f"{'':20}{'computed':>25}{'sys.float_info':>25}")
print(f"{'epsilon':20}{epsilon:>25.6e}{sys.float_info.epsilon:>25.6e}")
print(f"{'UFL':20}{UFL:>25.6e}{sys.float_info.min:>25.6e}")
print(f"{'OFL':20}{OFL:>25.6e}{sys.float_info.max:>25.6e}")


## 9. Errors — Absolute and Relative

$$|\theta - \hat\theta| \qquad \text{and} \qquad \frac{|\theta - \hat\theta|}{|\theta|} \; (\theta \neq 0)$$

In [ ]:
def errors(theta, theta_hat):
    abs_err = abs(theta - theta_hat)
    rel_err = abs_err / abs(theta) if theta != 0 else float('nan')
    print(f"theta       = {theta}")
    print(f"theta_hat   = {theta_hat}")
    print(f"absolute error = {abs_err}")
    print(f"relative error = {rel_err}")
    print()


# Example from the handout
errors(1_000_000, 1_000_001)

# Same absolute error, very different scale
errors(2, 3)


**Extension:** the same absolute error (5 points) means very different things depending on context — a football score prediction that's off by 5 points matters far more in a one-score game than in a blowout.

In [ ]:
# Extension: a win-probability model's predicted final score vs. the actual score
# Close game: predicted margin 3, actual margin 8  (absolute error 5)
errors(3, 8)

# Blowout: predicted margin 40, actual margin 45  (same absolute error, 5)
errors(40, 45)


Notice: both pairs above have the *same* absolute error (5), but very different **relative** errors. This is why relative error is often the more meaningful measure — being "off by 5" means something very different in a close game than in a blowout, and the same idea applies to revenue forecasts, stat projections, or any estimate compared against an actual outcome.

## 10. *(Optional)* Arbitrary-Precision Arithmetic — Exact Sums for Business

Floats round. For most scientific computing that's an acceptable trade-off for speed —
but for **exact currency totals**, small rounding errors can silently accumulate.

This section is a self-paced extension, especially relevant for **business majors**
working with financial ledgers, and is not part of the required 70-minute lecture flow.

In [ ]:
# Summing many small transactions as floats vs. as exact Decimals.
# Note: n = 10 is intentional here -- it reproduces the classic "0.1 added
# ten times is not exactly 1.0" drift in a way that's easy to see. Try
# larger values of n_transactions afterward; the drift doesn't disappear,
# it just moves to a different decimal place.
transaction = 0.10  # ten cents
n_transactions = 10

float_total = 0.0
for _ in range(n_transactions):
    float_total += transaction

decimal_total = sum(Decimal("0.10") for _ in range(n_transactions))

print(f"Float total   (n={n_transactions}): {float_total!r}")
print(f"Decimal total (n={n_transactions}): {decimal_total}")
print(f"Expected: exactly {n_transactions * 10}/100 = {n_transactions/10}")
print()
print("Float total == 1.0 ?", float_total == 1.0)
print("Difference from exact:", float_total - 1.0)


Also useful for exact fractions — e.g., betting or payout odds expressed as a ratio rather than a rounded decimal:

In [ ]:
# Fraction keeps an exact ratio instead of a rounded decimal
odds = Fraction(3, 7)
print("Exact odds as a fraction:", odds)
print("As a rounded float:      ", float(odds))


## 11. Closing: Resolving the Opening Puzzle

Revisit Section 1. Using what we now know about floating-point storage, rounding, and machine epsilon, run the cell below, then answer the question in the markdown cell that follows.

In [ ]:
for n in [10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    diff = S_forward(n) - s_reverse(n)
    print(f"n = {n:>10,}   S(n) - s(n) = {diff:.6e}")


**Your explanation:**

*(Double-click this cell to edit. In your own words, explain why $S(n)$ and $s(n)$ differ when computed on a computer, even though $S(n) = s(n)$ exactly in theory. Hint: think about the relative size of the running total compared to the next term being added, and what happens to small terms once the total gets large.)*

---
